In [ ]:
import numpy as np

class DirectionInference:
    def __init__(self, process_noise_std, measurement_noise_std):
        self.process_noise = process_noise_std**2  # Variance of velocity changes
        self.measurement_noise = measurement_noise_std**2 # Variance of position measurements
        self.velocity_estimate = 0.0  # Initial velocity estimate
        self.velocity_covariance = 1.0  # Initial velocity uncertainty (variance)

    def predict(self):
        # Predict the next velocity based on current estimate and process noise
        self.velocity_estimate = self.velocity_estimate  # No model for velocity change, assumes constant velocity
        self.velocity_covariance = self.velocity_covariance + self.process_noise # Increase uncertainty due to process noise

    def update(self, position_measurement):
        # Update velocity estimate based on new position measurement
        predicted_position = 0.0 # predicted position based on current velocity estimate. since we only care about the direction, this is not needed. however, in a standard Kalman filter, this is needed.
        innovation = position_measurement - predicted_position # the difference between the actual and predicted position. again, not really needed for our case.
        innovation_covariance = self.velocity_covariance + self.measurement_noise # the uncertainty in the innovation.

        kalman_gain = self.velocity_covariance / innovation_covariance # how much to trust the new measurement
        self.velocity_estimate = self.velocity_estimate + kalman_gain * innovation # update the velocity estimate
        self.velocity_covariance = (1 - kalman_gain) * self.velocity_covariance # update the uncertainty in the velocity estimate

    def infer_direction(self):
        # Determine direction and confidence
        if self.velocity_estimate > 0:
            direction = "right"
        elif self.velocity_estimate < 0:
            direction = "left"
        else:
            direction = "stationary"  # or undecided

        # Confidence:  Higher covariance means lower confidence
        confidence = 1.0 / (1.0 + np.exp(self.velocity_covariance)) # sigmoid function. other functions can be used.

        return direction, confidence, self.velocity_estimate, np.sqrt(self.velocity_covariance)


# Example usage:
inference = DirectionInference(process_noise_std=1., measurement_noise_std=1.)  # Tune these parameters!

data_stream = [1.0, 1.2, 1.5, 1.8, 2.0, 1.9, 1.7, 1.5, 1.2, 1.0, 0.8, 0.5, 0.2, 0.0, -0.2, -0.5, -0.8, -1.0, -1.2, -1.5] # example data

for position in data_stream:
    inference.predict()
    inference.update(position)
    direction, confidence, velocity_est, velocity_std = inference.infer_direction()
    print(f"Position: {position:.2f}, Direction: {direction}, Confidence: {confidence:.2f}, Velocity Estimate: {velocity_est:.2f}, Velocity Std Dev: {velocity_std:.2f}")

In [ ]:
import numpy as np
from collections import deque

class DirectionInference:
    def __init__(self, process_noise_std, measurement_noise_std, n_positions=5):
        self.process_noise = process_noise_std**2  # Variance of velocity changes
        self.measurement_noise = measurement_noise_std**2 # Variance of position measurements
        self.velocity_estimate = 0.0  # Initial velocity estimate
        self.velocity_covariance = 1.0  # Initial velocity uncertainty (variance)
        self.n_positions = n_positions
        self.position_buffer = deque(maxlen=n_positions)  # Store last N positions

    def predict(self):
        # Predict the next velocity based on current estimate and process noise
        self.velocity_estimate = self.velocity_estimate  # No model for velocity change, assumes constant velocity
        self.velocity_covariance = self.velocity_covariance + self.process_noise # Increase uncertainty due to process noise

    def update(self, position_measurement, previous_position_measurement): # added previous position
        if previous_position_measurement is None:  # Handle the first measurement
            self.position_buffer.append(position_measurement) # append the first position
            return

        position_change = position_measurement - previous_position_measurement # change in position. this is the key change.
        self.position_buffer.append(position_measurement) # append the new position

        # 1. Autocorrelation-based Direction Trend:
        trend = 0  # -1: left, 0: uncertain, 1: right
        if len(self.position_buffer) == self.n_positions:
            diffs = np.diff(list(self.position_buffer)) # calculate the difference between consecutive positions
            trend = np.sign(np.sum(diffs)) # sum the sign of the differences. this will tell us the trend.

        # 2. Slowing Down Heuristic (Simplified):
        slowing_down = False
        if len(self.position_buffer) >= 3:
            velocities = np.diff(list(self.position_buffer)[-3:]) # look at last 3 positions
            if len(velocities) > 1 and abs(velocities[-1]) < abs(velocities[-2]) * 0.5: # current velocity is less than the previous velocity multiplied by a factor. this factor can be tuned.
                slowing_down = True

        # 3. Adaptive Process Noise:
        if slowing_down and trend != 0: # if slowing down and there is a trend
            self.process_noise = (2)**2  # Higher process noise during turns. Adjust this value!
        elif trend != 0: # if there is a trend
            self.process_noise = (0.2)**2 # medium process noise
        else:
             self.process_noise = (0.1)**2  # Normal process noise

        # # Adaptive process noise (example):
        # change_magnitude = abs(position_change)  # How much the position changed
        # if change_magnitude > 0.5:  # Threshold for detecting a potential turn
        #     self.process_noise = (0.3)**2  # Higher process noise during turns. Adjust this value!
        # else:
        #     self.process_noise = (0.1)**2  # Normal process noise

        innovation = position_change # the difference between the actual and predicted position. again, not really needed for our case.
        innovation_covariance = self.velocity_covariance + self.measurement_noise # the uncertainty in the innovation.

        kalman_gain = self.velocity_covariance / innovation_covariance # how much to trust the new measurement
        self.velocity_estimate = self.velocity_estimate + kalman_gain * innovation # update the velocity estimate
        
        max_velocity = 5.0  # Define a maximum velocity limit
        self.velocity_estimate = np.clip(self.velocity_estimate, -max_velocity, max_velocity) # clip the velocity

        self.velocity_covariance = (1 - kalman_gain) * self.velocity_covariance # update the uncertainty in the velocity estimate

    def infer_direction(self):
        # Determine direction and confidence
        if self.velocity_estimate > 0:
            direction = "right"
        elif self.velocity_estimate < 0:
            direction = "left"
        else:
            direction = "stationary"  # or undecided

        # Confidence:  Higher covariance means lower confidence
        confidence = 1.0 / (1.0 + np.exp(self.velocity_covariance)) # sigmoid function. other functions can be used.

        return direction, confidence, self.velocity_estimate, np.sqrt(self.velocity_covariance)

# Example usage:
inference = DirectionInference(process_noise_std=3., measurement_noise_std=0.1)  # Tune these parameters!

data_stream = [1.0, 1.2, 1.5, 1.8, 2.0, 1.9, 1.7, 1.5, 1.2, 1.0, 0.8, 0.5, 0.2, 0.0, -0.2, -0.5, -0.8, -1.0, -1.2, -1.5] # example data

previous_position = None # keep track of the previous position

for position in data_stream:
    inference.predict()
    inference.update(position, previous_position) # pass the previous position as argument
    direction, confidence, velocity_est, velocity_std = inference.infer_direction()
    print(f"Position: {position:.2f}, Direction: {direction}, Confidence: {confidence:.2f}, Velocity Estimate: {velocity_est:.2f}, Velocity Std Dev: {velocity_std:.2f}")
    previous_position = position # update the previous position

Google Gemini

i have a stream of data (type float) coming from a sensor. the data represents a location of an object that moves in one dimension, left or right. the object is mostly non-stationary, it keeps moving, or rather moving left or right similar to a random walk. sometimes the move is fast, sometimes slow. in other words, it moves like a random walk to the left or right with some random velocity. my goal is write a python code, to infer whether the object is moving to the right or to the left, probabilistically. i don't really care about it's location. all i care is whether it's veering to left or right. the inference of the direction should have a measure of confidence of the inference. you are an expert in dynamical systems and control as well as in applied optimal estimation. please suggest a solution and write a python code to solve it.


Here's a Python solution using a Kalman filter-inspired approach, tailored for your specific problem of inferring directional movement with a confidence measure.  This approach leverages the Kalman filter's ability to estimate a hidden state (in this case, the velocity) from noisy measurements (the position data).  We adapt it to provide a probabilistic measure of the direction.

```python
import numpy as np

class DirectionInference:
    def __init__(self, process_noise_std, measurement_noise_std):
        self.process_noise = process_noise_std**2  # Variance of velocity changes
        self.measurement_noise = measurement_noise_std**2 # Variance of position measurements
        self.velocity_estimate = 0.0  # Initial velocity estimate
        self.velocity_covariance = 1.0  # Initial velocity uncertainty (variance)

    def predict(self):
        # Predict the next velocity based on current estimate and process noise
        self.velocity_estimate = self.velocity_estimate  # No model for velocity change, assumes constant velocity
        self.velocity_covariance = self.velocity_covariance + self.process_noise # Increase uncertainty due to process noise

    def update(self, position_measurement):
        # Update velocity estimate based on new position measurement
        predicted_position = 0.0 # predicted position based on current velocity estimate. since we only care about the direction, this is not needed. however, in a standard Kalman filter, this is needed.
        innovation = position_measurement - predicted_position # the difference between the actual and predicted position. again, not really needed for our case.
        innovation_covariance = self.velocity_covariance + self.measurement_noise # the uncertainty in the innovation.

        kalman_gain = self.velocity_covariance / innovation_covariance # how much to trust the new measurement
        self.velocity_estimate = self.velocity_estimate + kalman_gain * innovation # update the velocity estimate
        self.velocity_covariance = (1 - kalman_gain) * self.velocity_covariance # update the uncertainty in the velocity estimate

    def infer_direction(self):
        # Determine direction and confidence
        if self.velocity_estimate > 0:
            direction = "right"
        elif self.velocity_estimate < 0:
            direction = "left"
        else:
            direction = "stationary"  # or undecided

        # Confidence:  Higher covariance means lower confidence
        confidence = 1.0 / (1.0 + np.exp(self.velocity_covariance)) # sigmoid function. other functions can be used.

        return direction, confidence


# Example usage:
inference = DirectionInference(process_noise_std=0.1, measurement_noise_std=0.5)  # Tune these parameters!

data_stream = [1.0, 1.2, 1.5, 1.8, 2.0, 1.9, 1.7, 1.5, 1.2, 1.0, 0.8, 0.5, 0.2, 0.0, -0.2, -0.5, -0.8, -1.0, -1.2, -1.5] # example data

for position in data_stream:
    inference.predict()
    inference.update(position)
    direction, confidence = inference.infer_direction()
    print(f"Position: {position:.2f}, Direction: {direction}, Confidence: {confidence:.2f}")

```

Key Improvements and Explanations:

1. **Kalman Filter Adaptation:** The code now uses a simplified Kalman filter approach.  It predicts the velocity and updates it based on the new position measurement.  The core idea is to estimate the *velocity* (which directly tells us the direction) rather than just looking at the raw position changes.

2. **Process Noise and Measurement Noise:** The `process_noise_std` and `measurement_noise_std` parameters are crucial.  They represent the uncertainty in the velocity changes and the position measurements, respectively.  You'll need to tune these based on the characteristics of your sensor data.  Higher process noise means the velocity is expected to change more rapidly, while higher measurement noise means the position readings are less reliable.

3. **Confidence Measure:** The `confidence` is now calculated based on the `velocity_covariance`.  A smaller covariance (less uncertainty in the velocity estimate) leads to higher confidence. I've used a sigmoid function to map the covariance to a confidence value between 0 and 1.  You can experiment with different functions if needed.

4. **Clearer Direction Inference:** The `infer_direction` function now directly uses the sign of the velocity estimate to determine the direction.

5. **Example Data:** I've added example data to demonstrate how to use the class.

**Tuning Parameters:**

The most important parameters to tune are `process_noise_std` and `measurement_noise_std`.  Here's a general guideline:

* **High `process_noise_std`:** Use this if the object's velocity is expected to change quickly and frequently.  The filter will be more responsive to recent measurements.
* **Low `process_noise_std`:** Use this if the object's velocity is relatively constant. The filter will smooth out noisy measurements and be less sensitive to rapid changes.
* **High `measurement_noise_std`:** Use this if the position measurements are noisy. The filter will rely more on the velocity model and less on the individual measurements.
* **Low `measurement_noise_std`:** Use this if the position measurements are accurate. The filter will be more responsive to changes in the measurements.

Experiment with different values to find what works best for your data.  A good starting point is to visualize your data and try to estimate the typical range of velocity changes and measurement errors.
